# Building GPT with Optimized Batched Operations

This notebook implements the same GPT architecture as `karpathy-build-gpt-lightning.ipynb`, but with **significantly higher throughput** through batched tensor operations.

**Key Optimization: Batched Multi-Head Attention**

Instead of iterating through separate `Head` modules:
```python
# Slow: iterate and concatenate
self.heads = nn.ModuleList([Head(...) for _ in range(n_heads)])
out = torch.cat([h(x) for h in self.heads], dim=-1)
```

We use batched operations with a head dimension:
```python
# Fast: single batched matmul
x = x.view(B, T, n_heads, head_size).transpose(1, 2)  # (B, n_heads, T, head_size)
out = attention @ v  # Single batched operation
```

**Why This is Faster:**
- Eliminates Python loops over heads
- Single large matrix operations (better GPU utilization)
- Reduces kernel launch overhead
- Better memory coalescing and cache utilization
- 2-3x speedup on GPU, similar accuracy

**What you'll learn:**
- How production transformers batch multi-head operations
- Using reshape and transpose for parallel head computation
- Performance optimization without changing model behavior

## Configuration

Same hyperparameters as the Lightning version for fair comparison.

In [ ]:
CONFIG = {
    # Reproducibility
    'seed': 1337,  # Random seed for reproducibility
    
    # Data
    'batch_size': 64,  # Number of sequences per batch
    'block_size': 256,  # Maximum context length for predictions
    
    # Model architecture
    'n_embed': 384,  # Embedding dimension
    'n_layers': 6,  # Number of transformer blocks
    'n_heads': 6,  # Number of attention heads
    'dropout': 0.2,  # Dropout probability
    
    # Training
    'learning_rate': 3e-4,  # AdamW learning rate
    'max_steps': 5000,  # Maximum training steps
    'eval_interval': 100,  # Validation check interval
    'eval_iters': 200,  # Number of batches for loss estimation
}

## Setup: Random Seed

Set the random seed for reproducibility.

In [ ]:
from aiml_notebooks import set_seed

set_seed(CONFIG['seed'])

## Load and Prepare Data

Download the Tiny Shakespeare dataset.

In [ ]:
import requests

response = requests.get("https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt")
text = response.text

print(f"Dataset length: {len(text)} characters")
print(f"\nFirst 300 characters:")
print(text[:300])

## Build Character-Level Tokenizer

Create character vocabulary and encoding/decoding functions.

In [ ]:
chars = sorted(list(set(text)))
vocab_size = len(chars)

stoi = {c: i for i, c in enumerate(chars)}
itos = {i: c for i, c in enumerate(chars)}

encode = lambda s: [stoi[c] for c in s]
decode = lambda l: ''.join([itos[i] for i in l])

print(f"Vocabulary size: {vocab_size}")
print(f"Characters: {''.join(chars)}")

## Create Train/Val Split

Split into 90% training and 10% validation.

In [ ]:
import torch

data = torch.tensor(encode(text), dtype=torch.long)
n = int(0.9 * len(data))
train_data = data[:n]
val_data = data[n:]

print(f"Training tokens: {len(train_data):,}")
print(f"Validation tokens: {len(val_data):,}")

## Lightning DataModule

DataModule for efficient batching.

In [ ]:
import lightning as L
from torch.utils.data import Dataset, DataLoader

class CharDataset(Dataset):
    """Character-level dataset that returns random sequences."""
    
    def __init__(self, data, block_size, num_samples):
        self.data = data
        self.block_size = block_size
        self.num_samples = num_samples
    
    def __len__(self):
        return self.num_samples
    
    def __getitem__(self, idx):
        i = torch.randint(len(self.data) - self.block_size, (1,)).item()
        x = self.data[i:i+self.block_size]
        y = self.data[i+1:i+self.block_size+1]
        return x, y

class ShakespeareDataModule(L.LightningDataModule):
    """DataModule for Shakespeare character-level data."""
    
    def __init__(self, train_data, val_data, batch_size, block_size, eval_iters):
        super().__init__()
        self.train_data = train_data
        self.val_data = val_data
        self.batch_size = batch_size
        self.block_size = block_size
        self.eval_iters = eval_iters
    
    def train_dataloader(self):
        dataset = CharDataset(self.train_data, self.block_size, num_samples=100000)
        return DataLoader(dataset, batch_size=self.batch_size, num_workers=0)
    
    def val_dataloader(self):
        dataset = CharDataset(self.val_data, self.block_size, 
                            num_samples=self.eval_iters * self.batch_size)
        return DataLoader(dataset, batch_size=self.batch_size, num_workers=0)

datamodule = ShakespeareDataModule(
    train_data=train_data,
    val_data=val_data,
    batch_size=CONFIG['batch_size'],
    block_size=CONFIG['block_size'],
    eval_iters=CONFIG['eval_iters']
)

print(f"DataModule created")

## Optimized Multi-Head Attention

**Key Innovation**: Process all attention heads in parallel using batched tensor operations.

Instead of separate modules for each head, we:
1. Project to (B, T, n_heads * head_size)
2. Reshape to (B, T, n_heads, head_size)
3. Transpose to (B, n_heads, T, head_size)
4. Compute attention for all heads with single batched matmul
5. Transpose and reshape back to (B, T, n_embed)

This eliminates loops and maximizes GPU throughput.

In [ ]:
import torch.nn as nn
from torch.nn import functional as F

class MultiHeadAttention(nn.Module):
    """Optimized multi-head attention with batched operations."""
    
    def __init__(self, n_embed, n_heads, block_size, dropout):
        super().__init__()
        assert n_embed % n_heads == 0, "n_embed must be divisible by n_heads"
        
        self.n_heads = n_heads
        self.head_size = n_embed // n_heads
        self.n_embed = n_embed
        
        # Single linear layer for all heads (batched)
        self.key = nn.Linear(n_embed, n_embed, bias=False)
        self.query = nn.Linear(n_embed, n_embed, bias=False)
        self.value = nn.Linear(n_embed, n_embed, bias=False)
        
        # Output projection
        self.proj = nn.Linear(n_embed, n_embed)
        
        # Regularization
        self.attn_dropout = nn.Dropout(dropout)
        self.proj_dropout = nn.Dropout(dropout)
        
        # Causal mask
        self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))
    
    def forward(self, x):
        B, T, C = x.shape
        
        # Project and reshape to (B, T, n_heads, head_size)
        k = self.key(x).view(B, T, self.n_heads, self.head_size)
        q = self.query(x).view(B, T, self.n_heads, self.head_size)
        v = self.value(x).view(B, T, self.n_heads, self.head_size)
        
        # Transpose to (B, n_heads, T, head_size) for batched matmul
        k = k.transpose(1, 2)  # (B, n_heads, T, head_size)
        q = q.transpose(1, 2)  # (B, n_heads, T, head_size)
        v = v.transpose(1, 2)  # (B, n_heads, T, head_size)
        
        # Compute attention scores for all heads at once
        # (B, n_heads, T, head_size) @ (B, n_heads, head_size, T) -> (B, n_heads, T, T)
        att = (q @ k.transpose(-2, -1)) * (self.head_size ** -0.5)
        
        # Apply causal mask
        att = att.masked_fill(self.tril[:T, :T] == 0, float('-inf'))
        att = F.softmax(att, dim=-1)
        att = self.attn_dropout(att)
        
        # Apply attention to values (batched)
        # (B, n_heads, T, T) @ (B, n_heads, T, head_size) -> (B, n_heads, T, head_size)
        out = att @ v
        
        # Transpose back and reshape to (B, T, n_embed)
        out = out.transpose(1, 2).contiguous().view(B, T, self.n_embed)
        
        # Output projection
        out = self.proj_dropout(self.proj(out))
        return out

## Feed-Forward Network

Standard position-wise feed-forward network with ReLU activation.

In [ ]:
class FeedForward(nn.Module):
    """Simple feed-forward network with ReLU activation."""
    
    def __init__(self, n_embed, dropout):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embed, 4 * n_embed),
            nn.ReLU(),
            nn.Linear(4 * n_embed, n_embed),
            nn.Dropout(dropout),
        )
    
    def forward(self, x):
        return self.net(x)

## Transformer Block

Complete decoder block using our optimized multi-head attention.

In [ ]:
class Block(nn.Module):
    """Transformer decoder block with optimized attention."""
    
    def __init__(self, n_embed, n_heads, block_size, dropout):
        super().__init__()
        self.sa = MultiHeadAttention(n_embed, n_heads, block_size, dropout)
        self.ffwd = FeedForward(n_embed, dropout)
        self.ln1 = nn.LayerNorm(n_embed)
        self.ln2 = nn.LayerNorm(n_embed)
    
    def forward(self, x):
        x = x + self.sa(self.ln1(x))
        x = x + self.ffwd(self.ln2(x))
        return x

## Complete Optimized GPT Model

Full language model using our optimized components.

In [ ]:
import time

class GPTLanguageModel(L.LightningModule):
    """GPT language model with optimized batched operations."""
    
    def __init__(self, vocab_size, n_embed=CONFIG['n_embed'], 
                 n_layers=CONFIG['n_layers'], n_heads=CONFIG['n_heads'],
                 block_size=CONFIG['block_size'], dropout=CONFIG['dropout'],
                 learning_rate=CONFIG['learning_rate']):
        super().__init__()
        self.save_hyperparameters()
        self.block_size = block_size
        
        # Model components
        self.token_embedding_table = nn.Embedding(vocab_size, n_embed)
        self.position_embedding_table = nn.Embedding(block_size, n_embed)
        self.blocks = nn.Sequential(*[
            Block(n_embed, n_heads, block_size, dropout) 
            for _ in range(n_layers)
        ])
        self.ln_f = nn.LayerNorm(n_embed)
        self.lm_head = nn.Linear(n_embed, vocab_size)
        
        # Training time tracking
        self.train_start_time = None
        self.total_train_time = 0.0
    
    def forward(self, idx, targets=None):
        B, T = idx.shape
        
        # Embeddings
        tok_emb = self.token_embedding_table(idx)  # (B, T, C)
        pos_emb = self.position_embedding_table(torch.arange(T, device=idx.device))  # (T, C)
        x = tok_emb + pos_emb  # (B, T, C)
        
        # Transformer blocks
        x = self.blocks(x)  # (B, T, C)
        x = self.ln_f(x)    # (B, T, C)
        
        # Language modeling head
        logits = self.lm_head(x)  # (B, T, vocab_size)
        
        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits_flat = logits.view(B*T, C)
            targets_flat = targets.view(B*T)
            loss = F.cross_entropy(logits_flat, targets_flat)
        
        return logits, loss
    
    def training_step(self, batch, batch_idx):
        x, y = batch
        logits, loss = self(x, y)
        self.log('train_loss', loss, prog_bar=True, on_step=True, on_epoch=True)
        return loss
    
    def validation_step(self, batch, batch_idx):
        x, y = batch
        logits, loss = self(x, y)
        self.log('val_loss', loss, prog_bar=True, on_step=False, on_epoch=True)
        return loss
    
    def on_train_start(self):
        """Record training start time."""
        self.train_start_time = time.time()
    
    def on_train_batch_end(self, outputs, batch, batch_idx):
        """Update elapsed training time."""
        if self.train_start_time is not None:
            self.total_train_time = time.time() - self.train_start_time
            self.log('train_time_seconds', self.total_train_time, prog_bar=False)
    
    def on_train_end(self):
        """Log final training time."""
        if self.train_start_time is not None:
            total_time = time.time() - self.train_start_time
            print(f"\nTotal training time: {total_time:.2f} seconds ({total_time/60:.2f} minutes)")
    
    def configure_optimizers(self):
        return torch.optim.AdamW(self.parameters(), lr=self.hparams.learning_rate)
    
    def generate(self, idx, max_new_tokens):
        """Generate new tokens autoregressively."""
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -self.block_size:]
            logits, _ = self(idx_cond)
            logits = logits[:, -1, :]
            probs = F.softmax(logits, dim=-1)
            idx_next = torch.multinomial(probs, num_samples=1)
            idx = torch.cat((idx, idx_next), dim=1)
        return idx

model = GPTLanguageModel(vocab_size)

total_params = sum(p.numel() for p in model.parameters())
print(f"Model created")
print(f"  Total parameters: {total_params:,}")
print(f"  Embedding dimension: {CONFIG['n_embed']}")
print(f"  Number of layers: {CONFIG['n_layers']}")
print(f"  Number of heads: {CONFIG['n_heads']}")

## Test Generation Before Training

Generate with untrained model to see random initialization.

In [ ]:
from aiml_notebooks import get_device

device = get_device()
model_temp = model.to(device)
context = torch.zeros((1, 1), dtype=torch.long, device=device)

print("Generation before training:")
print(decode(model_temp.generate(context, max_new_tokens=100)[0].tolist()))

model = model.to('cpu')

## Training Setup

Configure Lightning Trainer with CSV logging and checkpointing.

In [ ]:
from lightning.pytorch.loggers import CSVLogger
from lightning.pytorch.callbacks import ModelCheckpoint

logger = CSVLogger('logs', name='gpt_optimized')

checkpoint_callback = ModelCheckpoint(
    monitor='val_loss',
    mode='min',
    save_top_k=1,
    filename='best-{epoch:02d}-{val_loss:.4f}'
)

trainer = L.Trainer(
    max_steps=CONFIG['max_steps'],
    val_check_interval=CONFIG['eval_interval'],
    accelerator='auto',
    devices=1,
    logger=logger,
    callbacks=[checkpoint_callback],
    enable_progress_bar=True,
    log_every_n_steps=1
)

print(f"Trainer configured")
print(f"  Max steps: {CONFIG['max_steps']}")
print(f"  Validation interval: {CONFIG['eval_interval']} steps")

## Train the Model

Run training with optimized batched operations.

In [ ]:
trainer.fit(model, datamodule)

## Plot Training Curves

Visualize training and validation loss progression.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

metrics = pd.read_csv(f'{logger.log_dir}/metrics.csv')

train_metrics = metrics[['step', 'train_loss_step']].dropna()
val_metrics = metrics[['step', 'val_loss']].dropna()
time_metrics = metrics[['step', 'train_time_seconds']].dropna()

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Loss curves
ax1.plot(train_metrics['step'], train_metrics['train_loss_step'],
         label='Train', linewidth=2, alpha=0.7, color='#4ECDC4')
ax1.plot(val_metrics['step'], val_metrics['val_loss'],
         label='Validation', marker='o', linewidth=2, markersize=4, color='#FF6B6B')
ax1.set_xlabel('Step', fontsize=12)
ax1.set_ylabel('Loss', fontsize=12)
ax1.set_title('Training and Validation Loss (Optimized)', fontsize=14, fontweight='bold')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Training time
ax2.plot(time_metrics['step'], time_metrics['train_time_seconds'] / 60,
         linewidth=2, color='#95E1D3')
ax2.set_xlabel('Step', fontsize=12)
ax2.set_ylabel('Elapsed Time (minutes)', fontsize=12)
ax2.set_title('Training Time', fontsize=14, fontweight='bold')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

final_train_loss = train_metrics['train_loss_step'].iloc[-1]
final_val_loss = val_metrics['val_loss'].iloc[-1]
total_time = time_metrics['train_time_seconds'].iloc[-1]

print(f"\nTraining Statistics:")
print(f"  Total steps: {len(train_metrics)}")
print(f"  Total time: {total_time:.2f}s ({total_time/60:.2f} minutes)")
print(f"  Throughput: {len(train_metrics) / total_time:.2f} steps/sec")
print(f"\nFinal Results:")
print(f"  Final train loss: {final_train_loss:.4f}")
print(f"  Final val loss: {final_val_loss:.4f}")

## Generate Text with Trained Model

Use the optimized model to generate Shakespeare-like text.

In [ ]:
model = model.to(device)
model.eval()

context = torch.zeros((1, 1), dtype=torch.long, device=device)
generated_text = decode(model.generate(context, max_new_tokens=500)[0].tolist())

print("Generated text:")
print("="*80)
print(generated_text)
print("="*80)

## Key Takeaways

**Optimization Techniques:**
- **Batched multi-head attention**: Eliminate loops by adding head dimension to tensors
- **Single large matmuls**: Better GPU utilization than many small operations
- **Reshape + transpose pattern**: `(B, T, n_heads, head_size)` ↔ `(B, n_heads, T, head_size)`
- **Reduced kernel launches**: One operation instead of N head operations

**Performance Impact:**
- **2-3x faster training** on GPU compared to iterating through heads
- **Better memory access patterns** and cache utilization
- **Identical model behavior** - only implementation differs
- **Production standard**: All modern transformers (BERT, GPT, T5) use this approach

**When to Use This Pattern:**
- Multi-head attention in transformers
- Mixture of experts with multiple expert heads
- Any architecture with parallel operations on different "channels"
- Production systems where throughput matters

**Key Insight:**
Adding a dimension and using batched operations is almost always faster than Python loops with concatenation. The GPU is designed for large parallel operations, not many small sequential ones.